In [ ]:
!npm install -g @anthropic-ai/claude-code

In [ ]:
# prompt: mount google drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install scanpy

In [ ]:
import scanpy as sc

# load the data

In [ ]:
datapath='/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/small_scale/raw_counts/filtered_feature_bc_matrix_S1lane1'

adata_full1=sc.read_10x_mtx(datapath,cache=True,gex_only=False)




In [ ]:
adata_full1.var

In [ ]:
adata_full1.obs

In [ ]:
datapath='/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/small_scale/raw_counts/filtered_feature_bc_matrix_S1lane2'

adata_full2=sc.read_10x_mtx(datapath,cache=True,gex_only=False)
adata_full2


In [ ]:
adata_full2.obs

In [ ]:
sum(adata_full2.obs_names.isin(adata_full1.obs_names))

In [ ]:
datapath='/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/small_scale/raw_counts/filtered_feature_bc_matrix_S2lane1'

adata_full3=sc.read_10x_mtx(datapath,cache=True,gex_only=False)
datapath='/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/small_scale/raw_counts/filtered_feature_bc_matrix_S2lane2'

adata_full4=sc.read_10x_mtx(datapath,cache=True,gex_only=False)


In [ ]:

adata_full = sc.concat(
        [adata_full1, adata_full2, adata_full3,adata_full4],
        label='lane_id',  # Adds a column to .obs indicating the source lane
        keys=['S1L1', 'S1L2','S2L1','S2L2'], # Labels for each lane
        index_unique='-' # Ensures unique cell barcodes across lanes
    )
adata_full.var=adata_full1.var

In [ ]:
adata_full.var

In [ ]:
adata_full2.var

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
feature_pd=adata_full.var
#feature_pd.columns=['gene_ids','genes' ,'feature_types']
feature_pd

In [ ]:
adata_full.var['genes']=feature_pd.index.to_list()

In [ ]:
guide_pd = feature_pd[feature_pd.feature_types=='Custom']
exp_pd = feature_pd[feature_pd.feature_types=='Gene Expression']

## save to file

In [ ]:
adata_full

In [ ]:
adata_full.write_h5ad('full_raw.h5ad')

## continue analysis

In [ ]:
exp_pd

In [ ]:
adata_exp=adata_full[:,adata_full.var.feature_types=='Gene Expression']
adata_exp

sc.pp.filter_cells(adata_exp, min_genes=200)
sc.pp.filter_genes(adata_exp, min_cells=3)


In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata_exp.var["mt"] = adata_exp.var_names.str.startswith("MT-")
# ribosomal genes
adata_exp.var["ribo"] = adata_exp.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata_exp.var["hb"] = adata_exp.var_names.str.contains("^HB[^(P)]")


In [ ]:
sc.pp.calculate_qc_metrics(adata_exp, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)


In [ ]:
adata_exp.var

In [ ]:
sc.pl.violin(
    adata_exp,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.scatter(adata_exp, "total_counts", "n_genes_by_counts", color="pct_counts_mt")


In [ ]:
sc.pp.filter_cells(adata_exp, min_genes=1000)
sc.pp.filter_genes(adata_exp, min_cells=3)
adata_exp = adata_exp[adata_exp.obs.pct_counts_mt < 20, :]
adata_exp.layers["counts"] = adata_exp.X.copy()

In [ ]:
adata_full.var.feature_types.value_counts()

In [ ]:
adata_full.X.shape

In [ ]:
guidef=adata_full[adata_exp.obs_names,adata_full.var.feature_types=='Custom']
#guidef.var['genes_corrected']=guidef.var['genes']
#guidef.var

guidef

# preprocessing

In [ ]:
import scanpy as sc

## restart from the saved h5ad files

In [ ]:
adata_full=sc.read_h5ad('full_raw.h5ad')
adata_full

In [ ]:

sc.pp.filter_cells(adata_full, min_genes=200)
sc.pp.filter_genes(adata_full, min_cells=3)

adata_full

In [ ]:
adata_full.var.feature_types.value_counts()

In [ ]:
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata_full.var["mt"] = adata_full.var_names.str.startswith("MT-")
# ribosomal genes
adata_full.var["ribo"] = adata_full.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata_full.var["hb"] = adata_full.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(adata_full, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)


In [ ]:
sc.pl.violin(
    adata_full,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

sc.pl.scatter(adata_full, "total_counts", "n_genes_by_counts", color="pct_counts_mt")


In [ ]:
adata_full = adata_full[adata_full.obs.pct_counts_mt < 30, :]
adata_full

In [ ]:
adata_guide = adata_full[:, adata_full.var.feature_types == "Custom"]
adata_full = adata_full[:, adata_full.var.feature_types == "Gene Expression"]

In [ ]:

sc.pp.normalize_total(adata_full)
sc.pp.log1p(adata_full)

In [ ]:

# calculate the highly variable genes. Note that we will change the n_top_genes to a larger number

sc.pp.highly_variable_genes(adata_full,n_top_genes=3000)



In [ ]:
adata_full.obs

In [ ]:

sc.tl.pca(adata_full)

sc.pl.pca_variance_ratio(adata_full, n_pcs=30, log=True)

In [ ]:




#sc.pl.pca(adata,color=["Batch"],dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
#  ncols=2,
#  size=2,
#)


sc.pp.neighbors(adata_full)

sc.tl.umap(adata_full)

#




In [ ]:
adata_full.obs

In [ ]:
sc.pl.umap(adata_full,color='n_genes',size=2)

## clustering

In [ ]:
!pip install leidenalg igraph louvain

In [ ]:
sc.tl.leiden(adata_full,flavor="igraph", n_iterations=2)



In [ ]:
#sc.tl.louvain(adata_full) # will crash

In [ ]:
adata_full

In [ ]:
sc.pl.umap(adata_full,color=["leiden",],size=2) # "louvain"

# assign cells to the highest expressed guides

In [ ]:
#guidef=adata_guide.var

In [ ]:
# remove all strings after "_" for guidef.var['genes']
adata_guide.var['genes_2'] = adata_guide.var['genes'].str.split('_').str[0]
adata_guide.var

In [ ]:
adata_guide.var.genes_2.value_counts()

In [ ]:
adata_guide.X

In [ ]:
# prompt: get the column index  of the highest value for each row of guidef.X

import numpy as np
gfxcp=adata_guide.X.toarray()
# Get the column index of the highest value for each row of guidef.X
highest_value_indices = np.argmax(gfxcp, axis=1)

# Print the indices
highest_value_indices

highest_values = np.max(gfxcp,axis=1)

# prompt: generate a [i, highest_value_indices[i]] duplex for each position i in highest_value_indices, and set the corresponding positions in gfxcp to zero

# Generate a duplex for each position i in highest_value_indices
# and set the corresponding positions in gfxcp to zero
duplex_indices = [(i, highest_value_indices[i]) for i in range(len(highest_value_indices))]
for i, j in duplex_indices:
    gfxcp[i, j] = 0

# Get the column index of the highest value for each row of guidef.X
second_highest_value_indices = np.argmax(gfxcp, axis=1)
second_highest_values = np.max(gfxcp,axis=1)



In [ ]:
print(highest_values[:20])
highest_value_indices[:20]

In [ ]:
print(second_highest_values[:20])
second_highest_value_indices[:20]

In [ ]:
adata_guide.X[0,:].toarray()

In [ ]:
import matplotlib.pyplot as plt

# Add a small constant to handle zero values for log scale
highest_values_log = highest_values + 1
second_highest_values_log = second_highest_values + 1

plt.figure(figsize=(8, 6))
plt.scatter(highest_values_log, second_highest_values_log, alpha=0.5, s=5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Highest Guide Count (log scale)')
plt.ylabel('Second Highest Guide Count (log scale)')
plt.title('Scatter Plot of Highest vs. Second Highest Guide Counts')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

In [ ]:
adata_guide.X[0,396]

In [ ]:
# prompt: add a "gene_ids" column guidef.obs as follows. For each row in guidef.X, calculate the column index for the highest and second highest values in this row of guidef.X. if the highest value is greater than 2 times second highest value, use the corresponding value in "gene_ids" column of guidef.var. Otherwise, assign "ambiguous". If the highest value is zero, assign "unassigned" label.

import numpy as np

adata_guide.obs["gene_ids"] = "unassigned"
adata_guide.obs["target_gene"]=None
adata_guide.obs["highest_guide_count"] = 0
# Iterate through each row of guidef.X
for i in range(adata_guide.X.shape[0]):
    if i % 10000 == 0:
        print(f"processing {i} cells...")

    h_value = highest_values[i]
    h_index=highest_value_indices[i]
    sh_value = second_highest_values[i]
    #sh_index = second_highest_value_indices[i]
    adata_guide.obs.iloc[i, adata_guide.obs.columns.get_loc("highest_guide_count")] = h_value
    if h_value == 0:
        adata_guide.obs.iloc[i, adata_guide.obs.columns.get_loc("gene_ids")] = "unassigned"
        adata_guide.obs.iloc[i, adata_guide.obs.columns.get_loc("target_gene")] = "unassigned"
    elif h_value > 1.2 * sh_value:
        adata_guide.obs.iloc[i, adata_guide.obs.columns.get_loc("gene_ids")] = adata_guide.var.iloc[h_index, adata_guide.var.columns.get_loc("gene_ids")]
        adata_guide.obs.iloc[i, adata_guide.obs.columns.get_loc("target_gene")] = adata_guide.var.iloc[h_index, adata_guide.var.columns.get_loc("genes_2")]
    else:
        adata_guide.obs.iloc[i, adata_guide.obs.columns.get_loc("gene_ids")] = "ambiguous"
        adata_guide.obs.iloc[i, adata_guide.obs.columns.get_loc("target_gene")] = "ambiguous"

adata_guide.obs



In [ ]:
adata_guide.obs.gene_ids.value_counts()

In [ ]:
adata_guide.var.gene_ids.value_counts().sort_values(ascending=False)

In [ ]:
adata_guide.var

In [ ]:
adata_guide.var.loc[adata_guide.var.gene_ids.str.contains('NANOG'),:]

In [ ]:
adata_guide.obs

In [ ]:
adata_full

In [ ]:
adata_full.obs['target_gene']=adata_guide.obs['target_gene']

In [ ]:
adata_guide.obs

In [ ]:
for var_id in adata_full.var_names[adata_full.var_names.str.contains('NANOG')]:
    sc.pl.umap(adata_full,color=[var_id],size=2)
#sc.pl.umap(adata,color=["RFX6"],size=2)

In [ ]:
adata_full.var


In [ ]:
adata_guide.var.genes_2.value_counts().to_dict()

In [ ]:
adata_full.obs

In [ ]:
adata_full.obs.target_gene.value_counts()

In [ ]:
perturbed_gene='EZH2'
neg_ctrl_group='ambiguous'


adata_full.obs['has_target_gene']=0

#adata.obs.iloc[cellular_index,adata.obs.columns.get_loc('has_target_gene')]=1
adata_full.obs.loc[adata_full.obs.target_gene==perturbed_gene,'has_target_gene']=1
#adata.obs.loc[adata.obs.target_gene==neg_ctrl_group,'has_target_gene']=2
adata_full.obs.loc[(~adata_full.obs.target_gene.isin([perturbed_gene,"unassigned"])),'has_target_gene']=2
adata_full.obs['has_target_gene'] = adata_full.obs['has_target_gene'].astype('category')

sc.pl.umap(adata_full,color=[perturbed_gene,'has_target_gene',],size=10)

In [ ]:
adata_full.obs.has_target_gene.value_counts()

In [ ]:
# prompt: plot the cumulative distribution for the NANOG expression of cells in 2 groups, depending on whether their has_target_gene column in adata.obs is 0 or 1.

import matplotlib.pyplot as plt
import seaborn as sns

# Extract NANOG expression for cells in group 0 and group 1
nanog_expression_group0 = adata_full[adata_full.obs['has_target_gene'] == 2, perturbed_gene].X.toarray().flatten()
nanog_expression_group1 = adata_full[adata_full.obs['has_target_gene'] == 1, perturbed_gene].X.toarray().flatten()

# Plot cumulative distribution
plt.figure(figsize=(8, 6))
sns.ecdfplot(data=nanog_expression_group0, label='neg ctrl group')
sns.ecdfplot(data=nanog_expression_group1, label='has_target_gene == 1')
plt.title(f'Cumulative Distribution of {perturbed_gene} Expression')
plt.xlabel(f'{perturbed_gene} Expression')
plt.ylabel('Cumulative Probability')
plt.legend()
plt.grid(True)
plt.ylim(0.0, 1)
plt.show()


In [ ]:
nanog_expression_group1.shape

In [ ]:
# prompt: perform k-s test for nanog_expression_group0 and nanog_expression_group1

from scipy.stats import ks_2samp

# Perform the K-S test
ks_statistic, p_value = ks_2samp(nanog_expression_group0, nanog_expression_group1)

# Print the results
print(f"K-S Statistic: {ks_statistic}")
print(f"P-value: {p_value}")


# compare the distribution of expression for cells with or without certain perturbation, not based on the highest assignment

In [ ]:
target_table=adata_full.obs.target_gene.value_counts()
target_table

In [ ]:
perturbed_gene='NANOG'
target_features=adata_guide.var.index[adata_guide.var.genes_2==perturbed_gene]
target_features_index = [adata_guide.var.index.get_loc(feature) for feature in target_features]
guide_matrix=adata_guide.X[:,target_features_index].toarray()
cellular_index=np.where(np.sum(guide_matrix, axis=1) > 5)[0]
guide_matrix
#target_features
cellular_index

In [ ]:
np.sum(guide_matrix,axis=1)

In [ ]:
# prompt: get row index of guide_matrix whose row sum is greater than 2

np.where(np.sum(guide_matrix, axis=1) > 2)[0]


In [ ]:
guide_matrix[cellular_index,:]

In [ ]:
adata_full

In [ ]:

adata_full.obs['has_target_gene']=0

adata_full.obs.iloc[cellular_index,adata_full.obs.columns.get_loc('has_target_gene')]=1


In [ ]:
#sc.pl.umap(adata_exp,color=[perturbed_gene,],size=2)

sc.pl.umap(adata_full[np.random.choice(adata_full.n_obs, size= int(adata_full.n_obs * 0.1), replace=False),:],color=[perturbed_gene,],size=10)

In [ ]:
adata_full.obs['has_target_gene'] = adata_full.obs['has_target_gene'].astype('category')

adata_full_target=adata_full[adata_full.obs.has_target_gene == 1,:]
sc.pl.umap(adata_full_target,color=['has_target_gene',],size=10)

adata_full_others=adata_full[adata_full.obs.has_target_gene == 0,:]
adata_full_others=adata_full_others[np.random.choice(adata_full_others.n_obs, size= int(adata_full_others.n_obs * 0.05), replace=False),:]

sc.pl.umap(adata_full_target.concatenate([adata_full_others]),color=['has_target_gene',],size=10)


In [ ]:
adata_full.obs.has_target_gene.value_counts()

In [ ]:
adata_full.obs['target_guide_expression']=np.log10(guide_matrix.sum(axis=1)+1.0)
sc.pl.umap(adata_full,color=['target_guide_expression',],size=10,color_map="Reds")

In [ ]:
# prompt: plot the cumulative distribution for the NANOG expression of cells in 2 groups, depending on whether their has_target_gene column in adata.obs is 0 or 1.

import matplotlib.pyplot as plt
import seaborn as sns

# Extract NANOG expression for cells in group 0 and group 1
nanog_expression_group0 = adata_full[adata_full.obs['has_target_gene'] == 0, perturbed_gene].X.toarray().flatten()
nanog_expression_group1 = adata_full[adata_full.obs['has_target_gene'] == 1, perturbed_gene].X.toarray().flatten()

# Plot cumulative distribution
plt.figure(figsize=(8, 6))
sns.ecdfplot(data=nanog_expression_group0, label='has_target_gene == 0')
sns.ecdfplot(data=nanog_expression_group1, label='has_target_gene == 1')
plt.title(f'Cumulative Distribution of {perturbed_gene} Expression')
plt.xlabel(f'{perturbed_gene} Expression')
plt.ylabel('Cumulative Probability')
plt.legend()
plt.grid(True)
plt.ylim(0.0, 1)
plt.show()


In [ ]:
nanog_expression_group1.shape

In [ ]:
# prompt: perform k-s test for nanog_expression_group0 and nanog_expression_group1

from scipy.stats import ks_2samp

# Perform the K-S test
ks_statistic, p_value = ks_2samp(nanog_expression_group0, nanog_expression_group1)

# Print the results
print(f"K-S Statistic: {ks_statistic}")
print(f"P-value: {p_value}")


## do all genes together

In [ ]:
target_table=adata_full.obs.target_gene.value_counts()
target_table

In [ ]:

#perturbed_gene='REST'
for perturbed_gene in target_table.index:
    target_features=adata_guide.var.index[adata_guide.var.genes_2==perturbed_gene]
    target_features_index = [adata_guide.var.index.get_loc(feature) for feature in target_features]
    guide_matrix=adata_guide.X[:,target_features_index].toarray()
    cellular_index=np.where(np.sum(guide_matrix, axis=1) > 5)[0]
    guide_matrix
    #target_features
    cellular_index

    adata_full.obs['has_target_gene']=0

    adata_full.obs.iloc[cellular_index,adata_full.obs.columns.get_loc('has_target_gene')]=1

    # Extract NANOG expression for cells in group 0 and group 1
    if perturbed_gene not in adata_full.var_names:
        continue
    nanog_expression_group0 = adata_full[adata_full.obs['has_target_gene'] == 0, perturbed_gene].X.toarray().flatten()
    nanog_expression_group1 = adata_full[adata_full.obs['has_target_gene'] == 1, perturbed_gene].X.toarray().flatten()


    from scipy.stats import ks_2samp

    # Perform the K-S test
    ks_statistic, p_value = ks_2samp(nanog_expression_group0, nanog_expression_group1)

    # Print the results
    print(f"K-S Statistic: {ks_statistic}")
    print(f"{perturbed_gene} P-value: {p_value}")

    # Plot cumulative distribution
    plt.figure(figsize=(8, 6))
    sns.ecdfplot(data=nanog_expression_group0, label='has_target_gene == 0')
    sns.ecdfplot(data=nanog_expression_group1, label='has_target_gene == 1')
    plt.title(f' {perturbed_gene} p= {p_value}')
    plt.xlabel(f'{perturbed_gene} Expression')
    plt.ylabel('Cumulative Probability')
    plt.legend()
    plt.grid(True)
    plt.ylim(0.0, 1)
    plt.show()


In [ ]:
#raise Exception("stop here")

In [ ]:
#adata_guide.obs

# save to h5ad file

In [ ]:
adata_full.write_h5ad('ESC_TF_perturbseq_processed.h5ad')
adata_guide.write_h5ad('ESC_TF_perturbseq_processed_guides.h5ad')


In [ ]:
#adata

In [ ]:
#adata.obs

In [ ]:
#adata.obs.target_gene.value_counts()

In [ ]:
!cp ESC_TF_perturbseq_processed.h5ad '/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/'

!cp ESC_TF_perturbseq_processed_guides.h5ad '/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/TF-PerturbSeq/'


In [ ]:
import scanpy as sc
adata_r=sc.read_h5ad('ESC_TF_perturbseq_processed.h5ad')

In [ ]:
adata_r.obs